In [ ]:
pip install gradio

In [ ]:
import gradio as gr

In [ ]:
pip install langchain_community

In [ ]:
pip install langchain

In [ ]:
pip install transformers

In [ ]:
pip install pypdf

In [ ]:
import gradio as gr
from transformers import pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

pdf_context = ""

In [ ]:
def process_pdf(file_obj):
    """Extracts text from the uploaded PDF file."""
    global pdf_context
    if file_obj is None:
        return "❌ Error: No file uploaded."

    try:
        # Load PDF using the file path provided by Gradio
        loader = PyPDFLoader(file_obj.name)
        pages = loader.load()

        # Split text into manageable chunks
        text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks = text_splitter.split_documents(pages)

        # Join chunks into one searchable string
        pdf_context = " ".join([c.page_content for c in chunks])

        print(f"Success! Extracted {len(pdf_context)} characters.")
        return "✅ PDF processed successfully! You can now ask questions."
    except Exception as e:
        print(e)

def chat_logic(user_question, history):
    """Handles the Question & Answering logic."""
    global pdf_context

    # Safety check: if no PDF is uploaded
    if not pdf_context or pdf_context.strip() == "":
        history.append((user_question, "⚠️ Please upload and 'Process' a PDF first."))
        return "", history

    try:

        context_slice = pdf_context[:3000]

        result = qa_pipeline(question=user_question, context=context_slice)
        answer = result["answer"]

    except:
        print('Error')

    # Update history for the Gradio Chatbot component
    history.append((user_question, answer))

    # Return "" to clear the input textbox, and the updated history
    return "", history

# --------------------------
# 3. Gradio Interface Construction
# --------------------------

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 AI PDF Assistant")
    gr.Markdown("Upload a PDF and ask questions about its content. Uses DistilBERT (Local CPU).")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="1. Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("2. Process PDF", variant="primary")
            status_output = gr.Textbox(label="System Status", interactive=False)

        with gr.Column(scale=2):
            chatbot_display = gr.Chatbot(label="Conversation", height=400)
            question_input = gr.Textbox(label="3. Ask a Question", placeholder="e.g., What is the main topic?")
            submit_btn = gr.Button("Send Question")


    # When Process PDF is clicked:
    process_btn.click(
        fn=process_pdf,
        inputs=[file_input],
        outputs=[status_output]
    )

    # When Send Question is clicked (or Enter is pressed):
    submit_event = submit_btn.click(
        fn=chat_logic,
        inputs=[question_input, chatbot_display],
        outputs=[question_input, chatbot_display]
    )

    question_input.submit(
        fn=chat_logic,
        inputs=[question_input, chatbot_display],
        outputs=[question_input, chatbot_display]
    )

# --------------------------
# 4. Launch
# --------------------------
if __name__ == "__main__":
    demo.launch()